## 1. Install Dependencies

In [1]:
!pip -q install gradio google-generativeai openai-whisper fpdf2 python-docx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 22.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 6.8 MB/s eta 0:00:00


## 2. Configure Gemini API Key

In [2]:
GEMINI_API_KEY = "KEY"

## 3. Imports & Setup

In [3]:
import gradio as gr
import google.generativeai as genai
import whisper
from fpdf import FPDF
from docx import Document
import tempfile, os

genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel("gemini-2.5-flash")
whisper_model = whisper.load_model("base")

chat_history = []
transcript_store = ""


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 101MiB/s]


## 4. Backend Functions

In [4]:
cancel_requested = False

def cancel_operation():
    global cancel_requested
    cancel_requested = True

def show_cancel(file):
    if file:
        return gr.Button(visible=True)
    return gr.Button(visible=False)

def transcript_stats(text):
    words = len(text.split())
    minutes = max(1, round(words / 180))

    return f"""
            ### 📊 Lecture Analytics

            - Words: {words}
            - Study Time: {minutes} mins
            - Status: Ready
            """
def file_info(file):
    if file is None:
        return ("⚪ No file selected", gr.Button(visible=False))

    filename = os.path.basename(str(file))
    return (f"""✅ File Ready**Name:** {filename}""", gr.Button(visible=True))

def transcribe_file(file, progress=gr.Progress()):

    global cancel_requested
    cancel_requested = False
    progress(0.1, desc="Reading file")

    if cancel_requested:
        return "❌ Cancelled"

    progress(0.3, desc="Preparing audio")

    if cancel_requested:
        return "❌ Cancelled"

    progress(0.6, desc="Transcribing")
    result = whisper_model.transcribe(file)

    global transcript_store
    transcript_store = result["text"]

    if cancel_requested:
        return "❌ Cancelled"

    progress(1.0, desc="Completed")
    return result["text"]

def ask_gemini(prompt):
    return model.generate_content(prompt).text

def generate_notes(transcript, mode):
    prompt = f"""Create {mode} study notes from the lecture transcript:{transcript}"""
    notes = ask_gemini(prompt)
    return notes, notes

def generate_quiz(transcript, difficulty):
    prompt = f"""Create 10 {difficulty} MCQs from:{transcript}"""
    return ask_gemini(prompt)

def generate_flashcards(transcript):
    return ask_gemini("Create flashcards from:\n"+transcript)


def generate_mindmap(transcript):

    mermaid = ask_gemini(
        """ Generate ONLY Mermaid mindmap code.
            No explanations.
            Transcript:""" + transcript
        )

    return f"""
      <div class="mermaid">
      {mermaid}
      </div>

      <script type="module">
      import mermaid from
      'https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.esm.min.mjs';

      mermaid.initialize({{
          startOnLoad:true
      }});
      </script>
      """

def chat_with_lecture(message, history):
    prompt = f""" You are an AI tutor.
                  Answer ONLY from the lecture.
                  If answer is unavailable,
                  say:'This topic was not covered in the lecture.'
                  Lecture: {transcript_store}
                  Question: {message}
              """
    return ask_gemini(prompt)

def export_pdf(content):
    if not content:
        return (None, "❌ Generate Notes First")

    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)
    pdf.multi_cell(0,10,content.encode('latin-1','replace').decode('latin-1'))
    path = "LectureGPT_Notes.pdf"
    pdf.output(path)
    # return path
    return (path,"✅ PDF Generated Successfully")

# V2 patch applied

transcript_store = ""
cancel_requested = False

def cancel_operation():
    global cancel_requested
    cancel_requested = True

def transcript_stats(text):
    words = len(text.split()) if text else 0
    minutes = max(1, round(words / 180))
    return f"""### 📊 Lecture Analytics
- Words: {words}
- Study Time: {minutes} mins
- Status: Ready
"""


## 5. Gradio UI

In [6]:
css = """

    .gr-button-primary{
        background:#ff7b00 !important;
        border:none !important;
    }

    .gr-button-primary:hover{
        background:#ff9500 !important;
    }

    footer{
        display:none !important;
    }

"""

with gr.Blocks(css=css, title="LectureGPT Pro") as demo:

    gr.HTML("""
      <div style='text-align:center'>
        <h1>🎓 LectureGPT Pro </h1>
        <p style='font-size:18px;color:#888;'>
          Transform Any Lecture Into
          Notes • Quiz • Flashcards • Mind Maps • AI Tutor
        </p>
      </div>
      """)

    notes_state = gr.State("")

    with gr.Group():
      gr.Markdown("## 📁 Upload Lecture")
      lecture_file = gr.File(label="Drag & Drop Video / Audio")
      upload_status = gr.Markdown("Waiting for file...")

    cancel_btn = gr.Button("❌ Cancel Processing", visible=False, variant="stop")

    cancel_btn.click(cancel_operation)

    transcript_box = gr.Textbox(lines=15,label='Transcript')
    transcribe_btn = gr.Button('🎙 Generate Transcript', variant="primary")

    stats = gr.Markdown()

    transcribe_btn.click(
      transcribe_file,
      lecture_file,
      transcript_box
    ).then(
        transcript_stats,
        transcript_box,
        stats
    )

    lecture_file.change(
      file_info,
      lecture_file,
      [upload_status, cancel_btn]
    )


    with gr.Tab('📚 Notes'):
        note_mode = gr.Dropdown(
            ['Detailed','Exam','Revision'],
            value='Detailed'
        )
        notes_btn = gr.Button("Generate Notes", variant="primary")
        notes_output = gr.Markdown()

        def start_notes():
          return gr.Button(value="⏳ Generating...", interactive=False)

        def end_notes(notes):
            return (notes, gr.Button(value="Generate Notes", interactive=True))

        notes_btn.click(
            start_notes,
            outputs=notes_btn
        ).then(
            generate_notes,
            [transcript_box, note_mode],
            [notes_output, notes_state]
        ).then(
            end_notes,
            notes_output,
            [notes_output, notes_btn]
        )

    with gr.Tab('🧠 Quiz'):
        difficulty = gr.Dropdown(
            ['Easy','Medium','Hard'],
            value='Medium'
        )
        quiz_btn = gr.Button('Generate Quiz', variant="primary")

        quiz_output = gr.Markdown()

        def start_quiz():
          return gr.Button(value="⏳ Generating...", interactive=False)

        def end_quiz(quiz):
            return (quiz, gr.Button(value="Generate Quiz", interactive=True))

        quiz_btn.click(
            start_quiz,
            outputs=quiz_btn
        ).then(
            generate_quiz,
            [transcript_box, difficulty],
            quiz_output
        ).then(
            end_quiz,
            quiz_output,
            [quiz_output, quiz_btn]
        )

    with gr.Tab('🎴 Flashcards'):
        flash_btn = gr.Button('Generate Flashcards', variant="primary")
        flash_output = gr.Markdown()

        # flash_btn.click(
        #     generate_flashcards,
        #     transcript_box,
        #     flash_output
        # )
        def start_flash():
          return gr.Button(value="⏳ Generating...", interactive=False)

        def end_flash(flash):
            return (flash, gr.Button(value="Generate Flash Cards", interactive=True))

        flash_btn.click(
            start_flash,
            outputs=flash_btn
        ).then(
            generate_flashcards,
            transcript_box,
            flash_output
        ).then(
            end_flash,
            flash_output,
            [flash_output, flash_btn]
        )

    with gr.Tab('🗺 Mind Map'):
        mind_btn = gr.Button('Generate Mind Map', variant="primary")
        mind_output = gr.HTML()

        def start_mind():
          return gr.Button(value="⏳ Generating...", interactive=False)

        def end_mind(mind):
            return (mind, gr.Button(value="Generate Mind map", interactive=True))

        mind_btn.click(
            start_mind,
            outputs=mind_btn
        ).then(
            generate_mindmap,
            transcript_box,
            mind_output
        ).then(
            end_mind,
            mind_output,
            [mind_output, mind_btn]
        )

    with gr.Tab('💬 AI Tutor'):
        gr.ChatInterface(
            fn=chat_with_lecture,
            title='Ask Your Lecture'
        )

    with gr.Tab('📄 Export'):
        pdf_btn = gr.Button('Export PDF', variant="primary")
        pdf_status = gr.Markdown()
        pdf_file = gr.File(label="📥 Download Report")

        def start_pdf():
          return gr.Button(value="⏳ Generating...", interactive=False)

        def end_pdf(file_path):
            return (file_path, gr.Button(value="Export PDF", interactive=True))

        pdf_btn.click(
            start_pdf,
            outputs=pdf_btn
        ).then(
            export_pdf,
            notes_state,
            [pdf_file, pdf_status]
        ).then(
            end_pdf,
            pdf_file,
            [pdf_file, pdf_btn]
        )

demo.launch(share=True, debug=True)


/tmp/ipykernel_4781/1497860558.py:18: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css, title="LectureGPT Pro") as demo:
/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e0bde446b39249c3ec.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://e0bde446b39249c3ec.gradio.live
